# **📊 Notebook 1 — Ingesta de Datos Reales con yfinance**
## iDeSo · UNMSM · FISI

---

**Integrante responsable:** Machaca Ponce Sebastian Emanuel (25200159)

**Objetivo:** Descargar datos OHLCV reales de 5 activos financieros mineros con operaciones en
Perú usando la librería `yfinance`, calcular los indicadores técnicos **SMA, EMA, RSI, MACD y
Bandas de Bollinger**, y exportar el resultado a un archivo JSON (`datos_mercado.json`)
compatible con el archivo `ernesto_investing_dashboard_mercado.html`.

**Activos del estudio:**

| Ticker | Empresa | Bolsa |
|--------|---------|-------|
| FSM | Fortuna Silver Mines Inc. | NYSE |
| VOLCABC1.LM | Volcan Compañía Minera S.A.A. | BVL |
| ABX.TO | Barrick Gold Corporation | TSX |
| BVN | Compañía de Minas Buenaventura | NYSE |
| BHP | BHP Billiton Limited | NYSE |

**Salida:** Archivo `datos_mercado.json` con estructura compatible con
`ernesto_investing_dashboard_mercado.html`


## Módulo 0 — Configuración de Activos

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 0 — CONFIGURACIÓN DE ACTIVOS DEL ESTUDIO
# Tickers de empresas mineras con operaciones en Perú
# ═══════════════════════════════════════════════════════════════

# Semilla fija para reproducibilidad (datos de contingencia)
SEED = 42

# Lista de tickers (idéntica a la usada en los Notebooks 2, 3 y 4)
TICKERS = [
    'FSM',         # Fortuna Silver Mines — NYSE
    'VOLCABC1.LM', # Volcan Compañía Minera — BVL
    'ABX.TO',      # Barrick Gold Corporation — TSX
    'BVN',         # Compañía de Minas Buenaventura — NYSE
    'BHP'          # BHP Billiton Limited — NYSE
]

# Nombres completos y bolsa para el JSON de salida
NOMBRES_EMPRESAS = {
    'FSM':         'Fortuna Silver Mines Inc.',
    'VOLCABC1.LM': 'Volcan Compañía Minera S.A.A.',
    'ABX.TO':      'Barrick Gold Corporation',
    'BVN':         'Compañía de Minas Buenaventura S.A.A.',
    'BHP':         'BHP Billiton Limited'
}

BOLSAS = {
    'FSM':         'NYSE',
    'VOLCABC1.LM': 'BVL',
    'ABX.TO':      'TSX',
    'BVN':         'NYSE',
    'BHP':         'NYSE'
}

# Ventana temporal de descarga
PERIODO   = '6mo'   # Últimos 6 meses
INTERVALO = '1d'    # Velas diarias

# Archivo de salida (contrato de datos con el dashboard de mercado)
ARCHIVO_SALIDA = 'datos_mercado.json'

print(f'✅ Configuración cargada: {len(TICKERS)} tickers, periodo={PERIODO}, intervalo={INTERVALO}')


## Módulo 1 — Instalación e Importación de Librerías

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 1 — INSTALACIÓN DE LIBRERÍAS
# Ejecutar solo si las librerías no están instaladas en el entorno
# ═══════════════════════════════════════════════════════════════

!pip install yfinance ta --quiet
print('✅ Librerías instaladas correctamente.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 1 — IMPORTACIÓN DE LIBRERÍAS
# ═══════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings('ignore')

# Librerías de datos y cálculo numérico
import numpy as np
import pandas as pd
import json
import random
from datetime import datetime, timedelta

# Descarga de datos financieros reales
import yfinance as yf

# Indicadores técnicos financieros
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands

print('✅ Librerías importadas correctamente.')


## Módulo 2 — Descarga de Datos OHLCV con yfinance

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 2 — DESCARGA DE DATOS DE MERCADO CON YFINANCE
# Datos OHLCV reales de Yahoo Finance para los 5 tickers, con
# manejo de errores y datos de contingencia ante fallos de red
# ═══════════════════════════════════════════════════════════════

def generar_datos_contingencia(ticker: str, dias: int = 130) -> pd.DataFrame:
    """
    Genera una serie OHLCV sintética de respaldo si yfinance falla
    (bloqueo temporal de IP, símbolo no disponible, sin conexión, etc.).
    Garantiza que el pipeline nunca se detenga por un fallo puntual de red.
    """
    random.seed(hash(ticker) & SEED)
    np.random.seed(hash(ticker) % (2**32 - 1))

    precios_base = {'FSM': 6.50, 'VOLCABC1.LM': 0.22, 'ABX.TO': 22.00,
                     'BVN': 15.40, 'BHP': 55.00}
    precio = precios_base.get(ticker, 15.0)

    fechas, opens, highs, lows, closes, vols = [], [], [], [], [], []
    fecha_actual = datetime.today() - timedelta(days=int(dias * 1.45))

    while len(fechas) < dias:
        if fecha_actual.weekday() < 5:  # Solo días hábiles
            variacion = np.random.normal(0.0008, 0.018)
            cierre = round(precio * (1 + variacion), 2)
            apertura = round(precio * (1 + np.random.normal(0, 0.005)), 2)
            maximo = round(max(apertura, cierre) * (1 + abs(np.random.normal(0, 0.006))), 2)
            minimo = round(min(apertura, cierre) * (1 - abs(np.random.normal(0, 0.006))), 2)

            fechas.append(fecha_actual.strftime('%Y-%m-%d'))
            opens.append(apertura)
            highs.append(maximo)
            lows.append(minimo)
            closes.append(cierre)
            vols.append(int(np.random.randint(400_000, 2_500_000)))
            precio = cierre
        fecha_actual += timedelta(days=1)

    df = pd.DataFrame({'Open': opens, 'High': highs, 'Low': lows,
                        'Close': closes, 'Volume': vols},
                       index=pd.to_datetime(fechas))
    return df


def descargar_datos_ohlcv(tickers: list, periodo: str, intervalo: str) -> dict:
    """
    Descarga datos OHLCV reales de Yahoo Finance para una lista de tickers.

    Parámetros:
        tickers   : Lista de símbolos bursátiles
        periodo   : Ventana temporal (ej. '6mo')
        intervalo : Granularidad de las velas (ej. '1d')

    Retorna:
        Diccionario { ticker: DataFrame } con los datos OHLCV descargados
        (reales o, en su defecto, de contingencia).
    """
    datos_crudos = {}

    for ticker in tickers:
        print(f'   ⬇️  Descargando {ticker}...')
        try:
            data = yf.download(ticker, period=periodo, interval=intervalo,
                                multi_level_index=False, progress=False)

            if isinstance(data.columns, pd.MultiIndex):
                data.columns = data.columns.get_level_values(0)

            if data.empty:
                raise ValueError('DataFrame vacío retornado por yfinance.')

            data = data[['Open', 'High', 'Low', 'Close', 'Volume']].ffill().bfill()
            datos_crudos[ticker] = data
            print(f'      ✅ {ticker}: {len(data)} registros reales descargados.')

        except Exception as e:
            print(f'      ⚠️  Error al descargar {ticker} ({e}). Activando contingencia...')
            datos_crudos[ticker] = generar_datos_contingencia(ticker)
            print(f'      🛟 {ticker}: {len(datos_crudos[ticker])} registros de contingencia generados.')

    return datos_crudos


# Ejecutar descarga para los 5 tickers del estudio
datos_ohlcv = descargar_datos_ohlcv(TICKERS, PERIODO, INTERVALO)
print()
print(f'✅ Descarga finalizada para {len(datos_ohlcv)} de {len(TICKERS)} tickers.')


## Módulo 3 — Cálculo de Indicadores Técnicos

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 3 — CÁLCULO DE INDICADORES TÉCNICOS
# SMA, EMA, RSI, MACD y Bandas de Bollinger, requeridos por el
# dashboard de mercado (ernesto_investing_dashboard_mercado.html)
# ═══════════════════════════════════════════════════════════════

def calcular_indicadores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula el conjunto de indicadores técnicos exigidos por el dashboard:

      - SMA-20, SMA-50  (Medias Móviles Simples)
      - EMA-12, EMA-26  (Medias Móviles Exponenciales)
      - RSI-14          (Relative Strength Index)
      - MACD, Señal MACD, Histograma MACD
      - Bandas de Bollinger (superior, media, inferior)

    Parámetros:
        df : DataFrame OHLCV de un único ticker

    Retorna:
        DataFrame original + columnas de indicadores técnicos
    """
    df = df.copy()

    # Medias móviles simples
    df['sma_20'] = SMAIndicator(df['Close'], window=20).sma_indicator()
    df['sma_50'] = SMAIndicator(df['Close'], window=50).sma_indicator()

    # Medias móviles exponenciales
    df['ema_12'] = EMAIndicator(df['Close'], window=12).ema_indicator()
    df['ema_26'] = EMAIndicator(df['Close'], window=26).ema_indicator()

    # Índice de fuerza relativa
    df['rsi_14'] = RSIIndicator(df['Close'], window=14).rsi()

    # MACD (Convergencia/Divergencia de Medias Móviles)
    macd = MACD(df['Close'], window_slow=26, window_fast=12, window_sign=9)
    df['macd']        = macd.macd()
    df['macd_signal'] = macd.macd_signal()
    df['macd_hist']   = macd.macd_diff()

    # Bandas de Bollinger
    bb = BollingerBands(df['Close'], window=20, window_dev=2)
    df['bb_upper']  = bb.bollinger_hband()
    df['bb_middle'] = bb.bollinger_mavg()
    df['bb_lower']  = bb.bollinger_lband()

    # Relleno de los primeros valores (ventanas móviles sin suficiente historial)
    df = df.bfill().ffill()

    return df


# Aplicar el cálculo de indicadores a los 5 tickers descargados
datos_con_indicadores = {}
for ticker, df in datos_ohlcv.items():
    datos_con_indicadores[ticker] = calcular_indicadores(df)
    print(f'   📈 {ticker}: indicadores técnicos calculados ({len(datos_con_indicadores[ticker])} filas).')

print()
print('✅ Indicadores técnicos calculados para todos los activos.')


## Módulo 4 — Validación y Limpieza de Datos

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 4 — VALIDACIÓN Y LIMPIEZA PREVIA A LA EXPORTACIÓN
# Verifica ausencia de NaN y consistencia de longitudes antes
# de construir el contrato de datos JSON
# ═══════════════════════════════════════════════════════════════

def validar_dataframe(df: pd.DataFrame, ticker: str) -> bool:
    """
    Valida que un DataFrame esté libre de valores nulos y que todas
    las columnas requeridas por el dashboard estén presentes.
    """
    columnas_requeridas = [
        'Open', 'High', 'Low', 'Close', 'Volume',
        'sma_20', 'sma_50', 'ema_12', 'ema_26', 'rsi_14',
        'macd', 'macd_signal', 'macd_hist',
        'bb_upper', 'bb_middle', 'bb_lower'
    ]

    faltantes = [c for c in columnas_requeridas if c not in df.columns]
    if faltantes:
        print(f'   ❌ {ticker}: faltan columnas {faltantes}')
        return False

    nulos = df[columnas_requeridas].isnull().sum().sum()
    if nulos > 0:
        print(f'   ⚠️  {ticker}: {nulos} valores nulos detectados, aplicando relleno final.')
        df[columnas_requeridas] = df[columnas_requeridas].ffill().bfill()

    print(f'   ✅ {ticker}: validado, {len(df)} registros, 0 valores nulos.')
    return True


print('Validando integridad de los datos antes de la exportación...')
for ticker, df in datos_con_indicadores.items():
    validar_dataframe(df, ticker)

print()
print('✅ Validación de integridad completada para todos los tickers.')


## Módulo 5 — Transformación a Estructura JSON del Frontend

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 5 — TRANSFORMACIÓN A ESTRUCTURA JSON
# Convierte cada DataFrame a un diccionario serializable, con la
# misma estructura que ernesto_investing_dashboard_mercado.html
# espera consumir mediante fetch()
# ═══════════════════════════════════════════════════════════════

def dataframe_a_json(df: pd.DataFrame, ticker: str) -> dict:
    """
    Transforma el DataFrame de un ticker (OHLCV + indicadores) al
    diccionario serializable que conforma el contrato de datos.
    """
    return {
        'empresa': NOMBRES_EMPRESAS[ticker],
        'bolsa':   BOLSAS[ticker],
        'dates':   [d.strftime('%Y-%m-%d') for d in df.index],
        'open':    df['Open'].round(4).tolist(),
        'high':    df['High'].round(4).tolist(),
        'low':     df['Low'].round(4).tolist(),
        'close':   df['Close'].round(4).tolist(),
        'volume':  df['Volume'].astype(int).tolist(),
        'indicadores': {
            'sma_20':      df['sma_20'].round(4).tolist(),
            'sma_50':      df['sma_50'].round(4).tolist(),
            'ema_12':      df['ema_12'].round(4).tolist(),
            'ema_26':      df['ema_26'].round(4).tolist(),
            'rsi_14':      df['rsi_14'].round(2).tolist(),
            'macd':        df['macd'].round(4).tolist(),
            'macd_signal': df['macd_signal'].round(4).tolist(),
            'macd_hist':   df['macd_hist'].round(4).tolist(),
            'bb_upper':    df['bb_upper'].round(4).tolist(),
            'bb_middle':   df['bb_middle'].round(4).tolist(),
            'bb_lower':    df['bb_lower'].round(4).tolist(),
        }
    }


# Construir el diccionario "mercado" con todos los tickers
mercado_json = {}
for ticker, df in datos_con_indicadores.items():
    mercado_json[ticker] = dataframe_a_json(df, ticker)
    print(f'   🔄 {ticker}: convertido a estructura JSON.')

print()
print('✅ Transformación a JSON completada para todos los tickers.')


## Módulo 6 — Exportación a JSON y Validación de Compatibilidad

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MÓDULO 6 — EXPORTACIÓN A JSON (CONTRATO DE DATOS)
# Genera datos_mercado.json compatible con
# ernesto_investing_dashboard_mercado.html
# Este archivo es el "contrato de datos" entre el backend (Python)
# y el frontend (HTML/JS)
# ═══════════════════════════════════════════════════════════════

# Construir el objeto JSON final
json_salida = {
    'meta': {
        'sistema':          'Ernesto Investing AI — Dashboard de Mercado',
        'version':          '1.0',
        'generado_en':      datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'fuente':           'yfinance',
        'periodo':          PERIODO,
        'intervalo':        INTERVALO,
        'indicadores':      ['SMA', 'EMA', 'RSI', 'MACD', 'Bandas de Bollinger'],
        'tickers':          TICKERS
    },
    'mercado': mercado_json
}

# Exportar a archivo local
with open(ARCHIVO_SALIDA, 'w', encoding='utf-8') as f:
    json.dump(json_salida, f, ensure_ascii=False, indent=2)

print(f'✅ Archivo {ARCHIVO_SALIDA} generado correctamente.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# VALIDACIÓN DE COMPATIBILIDAD CON EL DASHBOARD DE MERCADO
# Verifica que el JSON cumpla el contrato de datos esperado por
# ernesto_investing_dashboard_mercado.html antes de entregarlo
# ═══════════════════════════════════════════════════════════════

def validar_compatibilidad_json(ruta_archivo: str, tickers_esperados: list) -> bool:
    """
    Replica, sobre el archivo ya exportado, las verificaciones de
    compatibilidad con el frontend:
      1. Claves obligatorias presentes en cada ticker.
      2. Longitudes consistentes entre 'dates' y el resto de series.
      3. Ausencia de valores nulos / no serializables.
    """
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        data = json.load(f)

    claves_obligatorias = ['dates', 'open', 'high', 'low', 'close', 'volume', 'indicadores']
    claves_indicadores = ['sma_20', 'sma_50', 'ema_12', 'ema_26', 'rsi_14',
                           'macd', 'macd_signal', 'macd_hist',
                           'bb_upper', 'bb_middle', 'bb_lower']

    todo_ok = True

    for ticker in tickers_esperados:
        if ticker not in data['mercado']:
            print(f'   ❌ {ticker}: no encontrado en el JSON.')
            todo_ok = False
            continue

        activo = data['mercado'][ticker]
        n_fechas = len(activo['dates'])

        # 1. Claves obligatorias
        faltantes = [k for k in claves_obligatorias if k not in activo]
        if faltantes:
            print(f'   ❌ {ticker}: faltan claves {faltantes}')
            todo_ok = False
            continue

        # 2. Longitudes consistentes
        longitudes = {
            'open': len(activo['open']), 'high': len(activo['high']),
            'low': len(activo['low']), 'close': len(activo['close']),
            'volume': len(activo['volume'])
        }
        inconsistentes = {k: v for k, v in longitudes.items() if v != n_fechas}
        if inconsistentes:
            print(f'   ❌ {ticker}: longitudes inconsistentes {inconsistentes} (esperado {n_fechas})')
            todo_ok = False
            continue

        faltantes_ind = [k for k in claves_indicadores if k not in activo['indicadores']]
        if faltantes_ind:
            print(f'   ❌ {ticker}: faltan indicadores {faltantes_ind}')
            todo_ok = False
            continue

        for k in claves_indicadores:
            if len(activo['indicadores'][k]) != n_fechas:
                print(f'   ❌ {ticker}: indicador {k} con longitud inconsistente.')
                todo_ok = False

        print(f'   ✅ {ticker}: {n_fechas} registros, estructura 100% compatible con el dashboard.')

    return todo_ok


resultado_validacion = validar_compatibilidad_json(ARCHIVO_SALIDA, TICKERS)
print()
if resultado_validacion:
    print('🟢 VALIDACIÓN EXITOSA: el archivo es 100% compatible con ernesto_investing_dashboard_mercado.html')
else:
    print('🔴 VALIDACIÓN FALLIDA: revisar las inconsistencias señaladas arriba.')


## Celda Final — Descarga del Archivo JSON

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA FINAL — DESCARGA DEL ARCHIVO JSON EN GOOGLE COLAB
# ═══════════════════════════════════════════════════════════════

try:
    # Intentar descarga automática (funciona en Google Colab)
    from google.colab import files
    files.download(ARCHIVO_SALIDA)
    print(f'✅ {ARCHIVO_SALIDA} descargado al computador.')
except ImportError:
    # Si no está en Colab, indicar la ruta local
    import os
    ruta = os.path.abspath(ARCHIVO_SALIDA)
    print(f'✅ Archivo guardado localmente en: {ruta}')

print()
print('═' * 60)
print('  NOTEBOOK 1 COMPLETADO — Ingesta de Datos de Mercado')
print('  Ernesto Investing AI · iDeSo · UNMSM · FISI')
print('═' * 60)
print(f'  📄 Archivo generado : {ARCHIVO_SALIDA}')
print(f'  🌐 Frontend destino : ernesto_investing_dashboard_mercado.html')
print(f'  👤 Integrante       : Machaca Ponce Sebastian Emanuel (25200159)')
print('═' * 60)
